# 03 · Pitch calibration: image ↔ pitch

Everything in metres (speeds, distances, xG, zones) goes through a homography **H** (pitch plane → image) per frame
(`football_eval/pitch.py`). The provider depends on the camera:

* **static** — fixed camera, one H from ≥ 4 clicked landmarks;
* **flow** — panning camera: H is propagated frame to frame with ORB features on the background (players masked out),
  re-anchored on frames where you clicked landmarks; drift grows between anchors;
* **ptz** — fixed-position pan/tilt/zoom camera model (`ptz.py`): pan / tilt / focal per frame, refined on the white lines;
* **keypoints** — a pose model that finds pitch keypoints in every frame (needs a trained model, see 08).

Here: (1) the quality of the clicked landmarks, (2) the provider over the window, (3) players in pitch coordinates.

In [ ]:
import sys, os, time, json, warnings
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "football_eval").is_dir())
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")
import cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.precision", 3); pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30); pd.set_option("display.max_rows", 60)
plt.rcParams.update({"figure.dpi": 90, "axes.spines.top": False, "axes.spines.right": False})
from football_eval import nbtools as nbt

P = nbt.load_params()            # notebooks/pipeline/params.yaml (or $FVE_NB_PARAMS)
ctx = nbt.build_context(P)       # config <- match setup, restricted to the window
cfg = ctx.cfg

In [ ]:
from football_eval.pitch import Pitch, Homography, build_provider, draw_pitch
from football_eval.perception import foot_points, pitch_roi, visible_near_y
pitch = Pitch(cfg.pitch.length, cfg.pitch.width)
provider = build_provider(cfg, pitch, ctx.info.fps)
if getattr(provider, "cam", None) is not None:
    pitch = provider.cam.pitch
print(f"pitch {pitch.length} x {pitch.width} m | mode '{cfg.pitch.homography}' -> provider {type(provider).__name__ if provider else None}")
if hasattr(provider, "anchors"):
    an = sorted(provider.anchors)
    print(f"{len(an)} anchor frames at t = {[round(a / ctx.info.fps, 1) for a in an]} s; window {ctx.windows}")
    inside = [a for a in an if any(w0 * ctx.info.fps <= a <= w1 * ctx.info.fps for w0, w1 in ctx.windows)]
    if cfg.pitch.homography == "flow" and not inside:
        print("[!] no anchor inside the window: a flow provider has no pose until it meets one -> put an anchor in the window")

## 1 · Clicked landmarks: fit and reprojection error
For each landmark frame of the setup: H fitted on the clicks (RANSAC), then every click reprojected. Errors > ~5 px on a 1080p frame mean a wrong click or a wrong landmark name; a large error on one side of the pitch usually means lens distortion or a wrong pitch size.

In [ ]:
from football_eval.video import read_frame
L = pitch.landmarks()
frames_cal = (ctx.setup or {}).get("calibration", {}).get("landmark_frames", []) if ctx.setup else []
if not frames_cal and cfg.pitch.calibration_points:
    frames_cal = [{"t": P.probe_t, "points": cfg.pitch.calibration_points}]
cal_rows, cal_vis = [], []
for lf in frames_cal:
    pts = {k: v for k, v in lf["points"].items() if k in L}
    if len(pts) < 4:
        continue
    img_pts = np.array(list(pts.values()), float); pit_pts = np.array([L[k] for k in pts])
    Hc = Homography.fit(img_pts, pit_pts)
    err = np.hypot(*(Hc.to_image(pit_pts) - img_pts).T)
    for k, e in zip(pts, err):
        cal_rows.append(dict(t=round(lf["t"], 1), landmark=k, error_px=round(float(e), 1)))
    try:
        im = read_frame(ctx.video, float(lf["t"]))
    except ValueError:
        continue
    im = nbt.draw_pitch_lines(im, Hc, pitch)
    for (u, v), (pu, pv) in zip(img_pts, Hc.to_image(pit_pts)):
        cv2.circle(im, (int(u), int(v)), 9, (0, 0, 255), 3); cv2.drawMarker(im, (int(pu), int(pv)), (0, 255, 0), cv2.MARKER_CROSS, 22, 3)
    cal_vis.append((im, f"t={lf['t']:.1f}s: {len(pts)} clicks, mean error {err.mean():.1f} px"))
cal = pd.DataFrame(cal_rows)
for im, title in cal_vis[:2]:
    nbt.show(im, title + "  (red = click, green = reprojected, yellow = pitch lines)")
cal.groupby("t").error_px.describe()[["count", "mean", "max"]] if len(cal) else "no clicked landmarks in the setup"

## 2 · The provider over the window
The people boxes of 02 mask the players out of the background features (flow). For a PTZ camera the camera is first tracked from its anchor up to the window, as the real loop does. Projected pitch lines on frames across the window show drift.

In [ ]:
dets = nbt.load_stage(ctx, "detections")
anchor = getattr(provider, "anchor_frame", None)
if anchor is not None and ctx.windows[0][0] > anchor / ctx.info.fps + 1.0:
    print(f"PTZ warm-up from {anchor / ctx.info.fps:.0f}s to {ctx.windows[0][0]:.0f}s ...")
    provider.warmup(ctx.video, anchor / ctx.info.fps, ctx.windows[0][0], cfg.video.analysis_fps)
rows, shots = [], []
pick = None
for k, (f, t, im) in enumerate(nbt.frames_in_window(ctx)):
    people = nbt.detections_from_table(dets, f)
    H = provider.get(f, im, people.xyxy) if provider is not None else None
    row = nbt.homography_table_row(H, f, t)
    M = getattr(provider, "last_image_motion", None)
    if M is not None:
        row.update({f"m{i}{j}": float(np.asarray(M)[i, j]) for i in range(3) for j in range(3)})
    row["vis_near_y"] = visible_near_y(H, pitch, im.shape)
    rows.append(row)
    if k % max(1, int(len(dets.frame.unique()) / 4)) == 0:
        shots.append((nbt.draw_pitch_lines(im, H, pitch), f"frame {f} t={t:.1f}s  {'H ok' if H is not None else 'no H'}"))
hom = pd.DataFrame(rows)
print(f"pose in {hom.has_h.mean():.0%} of the window's frames")
nbt.grid([s[0] for s in shots[:4]], [s[1] for s in shots[:4]], cols=2, size=7);

## 3 · Stability of H: where do the pitch corners land, frame by frame?
Smooth curves = camera motion; jumps = a bad estimate (re-anchor, more clicks, or a better provider).

In [ ]:
corners = np.array([[0, 0], [pitch.length, 0], [pitch.length / 2, pitch.width / 2]])
tr = []
for _, r in hom.iterrows():
    Hr = nbt.homography_from_row(r)
    tr.append(Hr.to_image(corners).ravel() if Hr is not None else np.full(6, np.nan))
tr = np.array(tr)
fig, axes = plt.subplots(1, 2, figsize=(14, 3.2))
for j, name in enumerate(["far-left corner", "far-right corner", "centre spot"]):
    axes[0].plot(hom.t, tr[:, 2 * j], label=name); axes[1].plot(hom.t, tr[:, 2 * j + 1], label=name)
axes[0].set_title("image u (px)"); axes[1].set_title("image v (px)"); axes[0].legend(); axes[1].set_xlabel("t (s)");

## 4 · Players on the pitch (probe frame)
Foot point = bottom-centre of the box (estimated when the box is cut by the bottom of the frame) → H⁻¹ → metres. People outside the lines (+ margin) are staff / crowd (04).

In [ ]:
f, t, img = nbt.probe_frame(ctx)
r = hom.iloc[(hom.frame - f).abs().argmin()]
H = nbt.homography_from_row(r)
people = nbt.detections_from_table(dets, int(r.frame))
foot, edge = foot_points(people.xyxy, img.shape)
xy = H.to_pitch(foot) if H is not None and len(foot) else np.full((len(foot), 2), np.nan)
fig, axes = plt.subplots(1, 2, figsize=(16, 5), gridspec_kw={"width_ratios": [1.6, 1]})
nbt.show(nbt.draw_pitch_lines(nbt.draw_boxes(img, people.xyxy), H, pitch), f"frame {int(r.frame)}", ax=axes[0])
draw_pitch(axes[1], pitch)
axes[1].scatter(xy[:, 0], xy[:, 1], c=["C3" if e else "k" for e in edge], s=25)
axes[1].set_title("feet in pitch coordinates (red = foot estimated)")
roi = pitch_roi(H, pitch, img.shape) if H is not None else None
print("detector ROI from this pose (next frame):", roi, "| visible near touchline y:", r.vis_near_y)

In [ ]:
nbt.save_stage(ctx, "homography", hom);